# Companion Notebook — Chapter 07: Python Ecosystem for AI
**Artificial Intelligence Learning Hub** — *Institut Teknologi Sumatera (ITERA)*  
**Penulis**: Aji Pamungkas Tri Nurcahyo  

Notebook ini menyediakan panduan interaktif langsung (*hands-on*) untuk menjelajahi empat pustaka pilar ekosistem AI Python:
1. **NumPy**: Komputasi numerik array berkinerja tinggi, vektorisasi, dan broadcasting.
2. **Pandas**: Manipulasi tabel data terstruktur, pembacaan file CSV, dan agregasi terkelompok.
3. **Matplotlib**: Pembuatan visualisasi data dan grafik analitis.
4. **Scikit-learn**: Pola antarmuka Estimator API (`fit` dan `predict`) serta pembagian data latih/uji.
5. **Reproducibility**: Pengaturan seed deterministik untuk eksperimen yang dapat diulang.

## 1. NumPy: Array, Vektorisasi, dan Broadcasting

In [1]:
import numpy as np

# Membuat matriks data sensor 4 sampel x 3 fitur (Suhu, Kelembaban, Getaran)
X = np.array([
    [22.0, 45.0, 0.15],
    [24.0, 50.0, 0.20],
    [21.5, 42.0, 0.12],
    [28.5, 55.0, 0.85]
])

# Vektor offset kalibrasi 1D berukuran (3,)
offsets = np.array([-1.0, 2.0, -0.05])

# Broadcasting otomatis mereplikasi offsets sepanjang baris matriks
X_calibrated = X + offsets
mean_features = X.mean(axis=0)

print("Matriks X (4x3):\n", X)
print("Offset Kalibrasi:", offsets)
print("Hasil Broadcasting (X + offsets):\n", X_calibrated)
print("Mean per Fitur (axis=0):", mean_features)

Matriks X (4x3):
 [[22.   45.    0.15]
 [24.   50.    0.2 ]
 [21.5  42.    0.12]
 [28.5  55.    0.85]]
Offset Kalibrasi: [-1.    2.   -0.05]
Hasil Broadcasting (X + offsets):
 [[21.   47.    0.1 ]
 [23.   52.    0.15]
 [20.5  44.    0.07]
 [27.5  57.    0.8 ]]
Mean per Fitur (axis=0): [24.   48.    0.33]


## 2. Pandas: Manipulasi DataFrame & Analisis CSV

In [2]:
from pathlib import Path
import pandas as pd

data_path = Path("../code/data/sensor_readings.csv")
df = pd.read_csv(data_path)

# Imputasi missing values dengan mean
df["temperature_c"] = df["temperature_c"].fillna(df["temperature_c"].mean())
df["vibration_mms"] = df["vibration_mms"].fillna(df["vibration_mms"].mean())

print(f"Dataset Sensor ({df.shape[0]} baris x {df.shape[1]} kolom):")
print(df.head())

print("\nRata-rata Suhu dan Getaran per Lokasi:")
print(df.groupby("location")[["temperature_c", "vibration_mms"]].mean().round(2))

Dataset Sensor (20 baris x 7 kolom):
  sensor_id            timestamp       location  temperature_c  humidity_pct  vibration_mms   status
0       S01  2026-11-05 08:00:00  Server-Room-A           22.4          45.2           0.12   NORMAL
1       S02  2026-11-05 08:00:00  Server-Room-B           23.1          44.8           0.15   NORMAL
2       S03  2026-11-05 08:00:00  Turbine-Bay-1           28.5          52.1           0.85   NORMAL
3       S04  2026-11-05 08:00:00  Turbine-Bay-2           31.2          55.0           1.42  WARNING
4       S05  2026-11-05 08:05:00  Server-Room-A           22.6          45.0           0.11   NORMAL

Rata-rata Suhu dan Getaran per Lokasi:
               temperature_c  vibration_mms
location                                   
Server-Room-A          23.50           0.12
Server-Room-B          23.22           0.16
Turbine-Bay-1          28.94           0.90
Turbine-Bay-2          33.40           1.94


## 3. Matplotlib: Visualisasi Analitis

In [3]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Tren Garis Suhu
ax1.plot(df.index, df["temperature_c"], color="#0288D1", marker="o", label="Suhu (°C)")
ax1.set_title("Tren Suhu Sepanjang Waktu")
ax1.set_xlabel("Indeks Pengamatan")
ax1.set_ylabel("Suhu (°C)")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend()

# Plot 2: Scatter Plot Suhu vs Getaran
colors = df["status"].map({"NORMAL": "green", "WARNING": "orange", "CRITICAL": "red"})
ax2.scatter(df["temperature_c"], df["vibration_mms"], c=colors, s=70, edgecolors="black")
ax2.set_title("Korelasi Suhu vs Getaran Mekanis")
ax2.set_xlabel("Suhu (°C)")
ax2.set_ylabel("Getaran (mm/s)")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("Dashboard 2 grafik berhasil digenerasi: Tren Suhu dan Sebaran Suhu vs Getaran.")

Dashboard 2 grafik berhasil digenerasi: Tren Suhu dan Sebaran Suhu vs Getaran.


## 4. Scikit-learn: Siklus Estimator API

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Memisahkan matriks fitur X dan target biner y (0=Normal, 1=Anomali)
X = df[["temperature_c", "vibration_mms"]].values
y = (df["status"] != "NORMAL").astype(int).values

# Split train/test secara deterministik
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standarisasi tanpa kebocoran data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Estimator API: fit dan predict
clf = LogisticRegression(random_state=42)
clf.fit(X_train_scaled, y_train)
preds = clf.predict(X_test_scaled)

print(f"Data Latih (Train) : {X_train.shape[0]} sampel")
print(f"Data Uji (Test)    : {X_test.shape[0]} sampel")
print(f"Target Test Aktual : {y_test}")
print(f"Prediksi Model     : {preds}")
print(f"Akurasi Prediksi   : {accuracy_score(y_test, preds)*100:.2f}%")

Data Latih (Train) : 14 sampel
Data Uji (Test)    : 6 sampel
Target Test Aktual : [0 1 0 0 0 0]
Prediksi Model     : [0 1 0 0 0 0]
Akurasi Prediksi   : 100.00%


## 5. Uji Reproduksibilitas Eksperimen

In [5]:
def compute_features(seed=42):
    np.random.seed(seed)
    W = np.random.randn(2, 2)
    return X @ W

run1 = compute_features(seed=42)
run2 = compute_features(seed=42)

is_match = np.array_equal(run1, run2)
print(f"Uji Seed Deterministik: Apakah Matriks Run 1 == Run 2? {is_match}")
print(f"Selisih Maksimum: {np.max(np.abs(run1 - run2))} (Hasil 100% Identik dan Tervalidasi Reproduktif)")

Uji Seed Deterministik: Apakah Matriks Run 1 == Run 2? True
Selisih Maksimum: 0.0 (Hasil 100% Identik dan Tervalidasi Reproduktif)
